# Controlled SDXL LoRA generation on Kaggle

Generate images from **one LoRA adapter per run** by default. Clean and protected
adapters can be run separately using the same seeds and generation settings.
Optionally configure multiple adapters to generate them together. This notebook generates and validates output files;
perceptual, semantic and identity evaluation belong in a separate workflow.

Enable a GPU and Internet, attach the adapter(s), run Step 1 once, **restart the
kernel**, then continue at Step 2. Use a separate Kaggle session from training
because the dependency pins differ. For offline model loading, attach both the
SDXL checkpoint and its Diffusers configuration/tokenizer snapshot; a checkpoint
alone is not sufficient. Dependency installation still needs package access.

Defaults retain the existing experiment: `a photo of ohwx person`, 100 images
per condition, seeds **42–141**, 512×512, 25 steps, CFG 7.5 and DPM-Solver++.
Set adapter paths and verify training provenance before running. A filename
does not establish whether an adapter was trained on clean or protected images.

## Step 1 — Install one fixed inference environment

Preserve Kaggle's installed PyTorch build. PEFT is explicitly installed for LoRA
loading. No xformers installation is needed; inference uses PyTorch attention.
Any package conflict stops installation. Restart after this cell and skip it
when continuing. Exact installed versions are recorded with every experiment.

In [ ]:
import importlib.metadata as metadata
import json
import subprocess
import sys
from pathlib import Path

if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError("Use a Kaggle Python 3.10–3.12 GPU environment for these pins.")
working = Path("/kaggle/working")
if not working.is_dir():
    raise RuntimeError("This notebook expects Kaggle paths.")
pins = {
    "diffusers": "0.35.1", "transformers": "4.54.1", "accelerate": "1.10.1",
    "peft": "0.17.1", "huggingface-hub": "0.34.4", "safetensors": "0.6.2"
}
constraints = []
for name in ("torch", "torchvision", "torchaudio"):
    try:
        constraints.append(f"{name}=={metadata.version(name)}")
    except metadata.PackageNotFoundError:
        if name == "torch":
            raise RuntimeError("Select a Kaggle GPU image with PyTorch installed.")
constraint_path = working / "generation_torch_constraints.txt"
constraint_path.write_text("\n".join(constraints) + "\n", encoding="utf-8")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--constraint", str(constraint_path),
    *[f"{name}=={version}" for name, version in pins.items()]
], check=True)
(working / "generation_package_pins.json").write_text(json.dumps(pins, indent=2), encoding="utf-8")
print("Restart the kernel now, then continue from Step 2.")

## Step 2 — Select one adapter and configure generation

Set `LORA_PATH` to your actual `.safetensors` filename or its exact absolute path
under `/kaggle/input`. Set `RUN_LABEL` to identify the condition, such as `clean`
or `protected_alpha_090`. **Only this adapter is required for this run.**
The default filename is an example, not a required naming convention.

To generate another condition later, change `LORA_PATH` and `RUN_LABEL`, then rerun
**Steps 2–6**. Each generation run creates a fresh output folder and ZIP. Download
each ZIP before ending the Kaggle session. You do not need to attach both adapters
at the same time or reinstall packages when switching within the same session.

For an optional multi-adapter run, replace `CONDITIONS = {RUN_LABEL: LORA_PATH}`
with a dictionary of all desired labels and paths. Every entry must refer to an
available file; remove entries for adapters you are not generating in this run.

For comparisons across separate runs, keep prompts, seeds, resolution, scheduler,
adapter weight, precision, software versions and hardware consistent. Use the same
base checkpoint used for fine-tuning. Set BASE_REVISION to the training run's recorded
Hub commit when available; otherwise `main` is resolved once per run and recorded.
Compare the saved experiment manifests to verify the base hash and settings match.
Training comparability across separate runs must be checked from their manifests;
the notebook can compare adapter metadata only for adapters loaded in the same run.


In [ ]:
import os
import sys
import json
import math
import re
import hashlib
import importlib.metadata as metadata
from pathlib import Path

RUN_LABEL = "clean"  # Change to e.g. "protected_alpha_090" for a protected adapter.
LORA_PATH = "lora_run_a.safetensors"  # Set your actual filename or exact Kaggle path.
CONDITIONS = {RUN_LABEL: LORA_PATH}
# Optional: replace CONDITIONS with multiple label/path entries to generate them together.
SEARCH_ROOT = Path("/kaggle/input")
LOCAL_CHECKPOINT = None  # Optional exact path to the SDXL base .safetensors.
LOCAL_CONFIG_DIR = None  # Optional Diffusers config/tokenizer snapshot directory.
BASE_REPO = "stabilityai/stable-diffusion-xl-base-1.0"
BASE_REVISION = "main"  # Prefer the commit recorded by the fine-tuning run.
LOCAL_FILES_ONLY = False
CPU_OFFLOAD = True
PROMPT = "a photo of ohwx person"
NEGATIVE_PROMPT = ""
NUM_IMAGES = 100
BASE_SEED = 42
GENERATION = {
    "prompt": PROMPT, "negative_prompt": NEGATIVE_PROMPT,
    "height": 512, "width": 512, "num_inference_steps": 25,
    "guidance_scale": 7.5, "num_images_per_prompt": 1,
}
LORA_WEIGHT = 1.0  # Adapter inference strength, distinct from CS-UAP alpha.
SCHEDULER_OPTIONS = {
    "algorithm_type": "dpmsolver++", "solver_order": 2, "solver_type": "midpoint",
    "use_karras_sigmas": False, "lower_order_final": True,
    "timestep_spacing": "linspace", "final_sigmas_type": "zero",
}

if not CONDITIONS:
    raise ValueError("Configure at least one adapter in CONDITIONS.")
if any(not re.fullmatch(r"[A-Za-z0-9_-]+", label) for label in CONDITIONS):
    raise ValueError("Condition labels must use letters, digits, underscores or hyphens.")
if not isinstance(NUM_IMAGES, int) or NUM_IMAGES < 1 or not isinstance(BASE_SEED, int):
    raise ValueError("NUM_IMAGES must be positive and BASE_SEED an integer.")
if not 0 <= BASE_SEED <= BASE_SEED + NUM_IMAGES - 1 < 2**63:
    raise ValueError("Seed sequence is outside the supported range.")
for key in ("height", "width"):
    if not isinstance(GENERATION[key], int) or GENERATION[key] < 64 or GENERATION[key] % 8:
        raise ValueError("Image dimensions must be integers >=64 and divisible by 8.")
if not isinstance(GENERATION["num_inference_steps"], int) or GENERATION["num_inference_steps"] < 1:
    raise ValueError("Inference steps must be a positive integer.")
if not PROMPT.strip() or not math.isfinite(GENERATION["guidance_scale"]) or GENERATION["guidance_scale"] <= 1:
    raise ValueError("This CFG protocol requires a nonempty prompt and guidance scale >1.")
if GENERATION["num_images_per_prompt"] != 1 or not math.isfinite(LORA_WEIGHT) or LORA_WEIGHT <= 0:
    raise ValueError("Use one image per call and a positive finite adapter weight.")
SEEDS = list(range(BASE_SEED, BASE_SEED + NUM_IMAGES))
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
print(f"Conditions: {list(CONDITIONS)}; {NUM_IMAGES} images each; seeds {SEEDS[0]}–{SEEDS[-1]}")

## Step 3 — Validate adapters and runtime

Ambiguous matches, identical adapter files, invalid tensors and non-SDXL metadata
stop the run. Missing training metadata cannot establish comparable training;
verify the original run manifests, image set, captions and checkpoint step separately.

In [ ]:
import torch
import diffusers
import transformers
import peft
import accelerate
from safetensors import safe_open
from safetensors.torch import load_file

pin_file = Path("/kaggle/working/generation_package_pins.json")
if not pin_file.is_file():
    raise RuntimeError("Complete Step 1 and restart the kernel first.")
pins = json.loads(pin_file.read_text(encoding="utf-8"))
PACKAGE_VERSIONS = {name: metadata.version(name) for name in (*pins, "torch", "numpy", "Pillow")}
if any(PACKAGE_VERSIONS[name] != version for name, version in pins.items()):
    raise RuntimeError("Installed packages differ from Step 1. Use a fresh session and reinstall.")
for module in (diffusers, transformers, peft, accelerate):
    if module.__version__ != pins[module.__name__]:
        raise RuntimeError("Loaded modules are stale. Restart the kernel.")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator and restart the kernel.")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True)
print("GPU:", torch.cuda.get_device_name(0))

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def resolve_adapter(value, root=SEARCH_ROOT):
    path = Path(value)
    if path.is_absolute():
        matches = [path] if path.is_file() else []
    else:
        matches = sorted(p for p in root.rglob(str(path)) if p.is_file())
    if len(matches) != 1:
        raise ValueError(f"Expected one match for {value!r}, found {len(matches)}: {matches}. Set an exact path.")
    if matches[0].suffix.lower() != ".safetensors":
        raise ValueError("Use safetensors adapters.")
    return matches[0].resolve()

ADAPTERS = {}
for label, value in CONDITIONS.items():
    path = resolve_adapter(value)
    state = load_file(str(path), device="cpu")
    if not state or not any("lora" in key.lower() and "unet" in key.lower() for key in state):
        raise ValueError(f"No U-Net LoRA tensors found in {path}")
    if any(not torch.isfinite(tensor).all().item() for tensor in state.values()):
        raise ValueError(f"Non-finite adapter tensors: {path}")
    with safe_open(str(path), framework="pt", device="cpu") as handle:
        training_metadata = handle.metadata() or {}
    base_version = training_metadata.get("ss_base_model_version", "")
    if base_version and "sdxl" not in base_version.lower():
        raise ValueError(f"Adapter metadata identifies a different model: {base_version}")
    ADAPTERS[label] = {"path": str(path), "sha256": sha256_file(path), "metadata": training_metadata}
    del state
if len({a["sha256"] for a in ADAPTERS.values()}) != len(ADAPTERS):
    raise ValueError("Two conditions point to identical adapter content. Check the mapping.")
comparison_fields = ("ss_network_dim", "ss_network_alpha", "ss_base_model_version",
                     "ss_resolution", "ss_seed", "ss_steps", "ss_max_train_steps")
for label, item in ADAPTERS.items():
    print(label, {field: item["metadata"].get(field, "unknown") for field in comparison_fields})
for field in comparison_fields:
    known = {a["metadata"][field] for a in ADAPTERS.values() if field in a["metadata"]}
    if field in {"ss_network_dim", "ss_network_alpha", "ss_seed", "ss_steps", "ss_max_train_steps"}:
        known = {float(value) for value in known}
    if len(known) > 1:
        raise ValueError(f"Training metadata differs across conditions for {field}: {known}")
print("Validated adapters:", {label: item["path"] for label, item in ADAPTERS.items()})

## Step 4 — Load one shared base pipeline

The original SDXL single-file checkpoint matches the fine-tuning notebook's model
format. Downloaded configuration, tokenizers and weights use one resolved revision.
The checkpoint is hashed without making another copy. The scheduler options,
negative prompt, watermark setting and adapter weight are explicit for every run.
FP16 inference retains the SDXL VAE's FP32 upcast for numerical stability.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

cache = "/kaggle/working/hf-cache"
resolved_revision = None
if LOCAL_CONFIG_DIR is None or LOCAL_CHECKPOINT is None:
    if LOCAL_FILES_ONLY:
        if not re.fullmatch(r"[0-9a-f]{40}", BASE_REVISION):
            raise ValueError("Offline cache loading requires an exact cached commit or both local model paths.")
        resolved_revision = BASE_REVISION
    else:
        resolved_revision = HfApi().model_info(BASE_REPO, revision=BASE_REVISION).sha
config_dir = Path(LOCAL_CONFIG_DIR) if LOCAL_CONFIG_DIR else Path(snapshot_download(
    BASE_REPO, revision=resolved_revision, cache_dir=cache, local_files_only=LOCAL_FILES_ONLY,
    allow_patterns=["*.json", "tokenizer/*", "tokenizer_2/*"]
))
checkpoint = Path(LOCAL_CHECKPOINT) if LOCAL_CHECKPOINT else Path(hf_hub_download(
    BASE_REPO, filename="sd_xl_base_1.0.safetensors", revision=resolved_revision,
    cache_dir=cache, local_files_only=LOCAL_FILES_ONLY
))
if not checkpoint.is_file() or not (config_dir / "model_index.json").is_file():
    raise FileNotFoundError("Provide a valid SDXL checkpoint and Diffusers configuration snapshot.")
print("Hashing base checkpoint...")
BASE_INFO = {"repo": BASE_REPO, "revision": resolved_revision,
    "checkpoint": str(checkpoint), "sha256": sha256_file(checkpoint), "config_dir": str(config_dir),
    "config_hashes": {str(p.relative_to(config_dir)): sha256_file(p)
                      for p in sorted(config_dir.rglob("*")) if p.is_file() and p.suffix in {".json", ".txt"}}}
# Release a previously loaded pipeline when rerunning this cell.
import gc
if "pipe" in globals():
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
pipe = StableDiffusionXLPipeline.from_single_file(
    str(checkpoint), config=str(config_dir), local_files_only=True,
    torch_dtype=torch.float16, use_safetensors=True, add_watermarker=False
)
pipe.vae.register_to_config(force_upcast=True)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, **SCHEDULER_OPTIONS)
SCHEDULER_CONFIG = dict(pipe.scheduler.config)
if CPU_OFFLOAD:
    pipe.enable_model_cpu_offload(gpu_id=0)
else:
    pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)
print("Shared SDXL pipeline ready. CPU offload:", CPU_OFFLOAD)

## Step 5 — Generate the selected adapter(s)

Each condition loads exactly one adapter, with errors surfaced if unloading fails.
A fresh CPU RNG is seeded for every image in every condition. DPM-Solver++ is reset
for each condition. Do not fuse adapters: unloading must leave the base weights intact.
CPU RNG and deterministic settings help repeatability; different hardware or software
versions can still change results. These outputs must be regenerated consistently
when changing the protocol or dependency stack.

Every execution creates a new experiment folder. Interrupted runs retain their status
and completed image records; rerunning starts a new experiment, without mixing results.
Non-finite decoded pixels stop generation before PNG conversion. File validation here
is operational checking, not a measure of CS-UAP effectiveness.

Scheduler metadata can contain non-finite defaults such as `lambda_min_clipped=-inf`.
The JSON writer records these as the strings `"-Infinity"`, `"Infinity"`, or `"NaN"`
while leaving the live numeric configuration unchanged. These strings are for logging;
convert them back to numbers before reusing an exported configuration. Decoded image
pixels must still be finite. If an older Step 5 failed while writing JSON, replace
that cell and rerun Steps 5 and 6 in the same session; no model reload is needed.


In [ ]:
import tempfile
import time
import numpy as np
from PIL import Image
from datetime import datetime, timezone
from tqdm.auto import tqdm

def json_metadata(value):
    """Copy metadata into JSON types; encode non-finite floats as explicit strings.

    These strings are for logging only, not scheduler reconstruction. The original
    in-memory scheduler configuration retains its numeric values.
    """
    if isinstance(value, dict):
        return {key: json_metadata(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_metadata(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_metadata(value.tolist())
    if isinstance(value, np.generic):
        return json_metadata(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        if np.isnan(value):
            return "NaN"
        return "Infinity" if value > 0 else "-Infinity"
    return value


def write_json(path, data):
    # DPM-Solver's lambda_min_clipped defaults to -inf, a valid scheduler setting
    # that JSON cannot represent as a number. Convert only the saved metadata.
    serialized = json.dumps(json_metadata(data), indent=2, ensure_ascii=False, allow_nan=False)
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(serialized, encoding="utf-8")
    temporary.replace(path)

def image_pixels(result):
    pixels = np.asarray(result.images)
    if pixels.shape != (1, GENERATION["height"], GENERATION["width"], 3):
        raise RuntimeError(f"Unexpected pipeline output shape: {pixels.shape}")
    if not np.isfinite(pixels).all():
        raise FloatingPointError("Non-finite generated pixels. Check precision and adapter weights.")
    if pixels.min() < 0 or pixels.max() > 1:
        raise RuntimeError("Decoded pixels are outside [0, 1].")
    return np.rint(pixels[0] * 255).astype(np.uint8)

def generate_condition(label, adapter, root):
    output_dir = root / label
    output_dir.mkdir()
    if any(pipe.get_list_adapters().values()):
        pipe.unload_lora_weights()
    if any(pipe.get_list_adapters().values()):
        raise RuntimeError("Previous adapters remain loaded.")
    if sha256_file(adapter["path"]) != adapter["sha256"]:
        raise RuntimeError("Adapter changed since validation. Rerun Step 3.")
    pipe.load_lora_weights(str(Path(adapter["path"]).parent),
        weight_name=Path(adapter["path"]).name, adapter_name="condition")
    pipe.set_adapters("condition", adapter_weights=LORA_WEIGHT)
    if pipe.get_active_adapters() != ["condition"]:
        raise RuntimeError("Unexpected active adapters.")
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(SCHEDULER_CONFIG)
    records = []
    for index, seed in enumerate(tqdm(SEEDS, desc=label)):
        generator = torch.Generator(device="cpu").manual_seed(seed)
        with torch.inference_mode():
            result = pipe(**GENERATION, generator=generator, output_type="np")
        pixels = image_pixels(result)
        filename = f"gen_{index:03d}.png"
        destination = output_dir / filename
        temporary = destination.with_suffix(".png.tmp")
        Image.fromarray(pixels).save(temporary, format="PNG")
        temporary.replace(destination)
        records.append({"index": index, "seed": seed, "file": filename,
            "sha256": sha256_file(destination), "adapter_sha256": adapter["sha256"]})
        write_json(output_dir / "images.json", records)
    pipe.unload_lora_weights()
    if any(pipe.get_list_adapters().values()):
        raise RuntimeError("Adapter cleanup failed.")
    return records

root = Path("/kaggle/working/generated")
root.mkdir(parents=True, exist_ok=True)
run_prefix = f"{next(iter(ADAPTERS))}_" if len(ADAPTERS) == 1 else "comparison_"
EXPERIMENT_DIR = Path(tempfile.mkdtemp(prefix=run_prefix, dir=root))
experiment = {
    "status": "running", "started_utc": datetime.now(timezone.utc).isoformat(),
    "generation": GENERATION.copy(), "seeds": SEEDS, "generator_device": "cpu",
    "scheduler": SCHEDULER_CONFIG, "lora_weight": LORA_WEIGHT, "watermark": False,
    "vae_force_upcast": True, "precision": "float16", "cpu_offload": CPU_OFFLOAD,
    "deterministic_algorithms": True, "tf32": False, "base": BASE_INFO,
    "adapters": ADAPTERS, "packages": PACKAGE_VERSIONS,
    "python": sys.version, "gpu": torch.cuda.get_device_name(0),
    "torch_cuda": torch.version.cuda, "conditions_completed": [],
}
manifest_path = EXPERIMENT_DIR / "experiment.json"
write_json(manifest_path, experiment)
start = time.monotonic()
try:
    for label, adapter in ADAPTERS.items():
        generate_condition(label, adapter, EXPERIMENT_DIR)
        experiment["conditions_completed"].append(label)
        write_json(manifest_path, experiment)
    experiment["status"] = "generated"
except BaseException as error:
    experiment["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
    experiment["error"] = f"{type(error).__name__}: {error}"
    raise
finally:
    experiment["finished_utc"] = datetime.now(timezone.utc).isoformat()
    experiment["seconds"] = time.monotonic() - start
    write_json(manifest_path, experiment)
print("Generation finished. Validate files in the next cell:", EXPERIMENT_DIR)

## Step 6 — Validate the selected outputs and package them

Require every planned image, seed and condition, verify PNG integrity, dimensions,
and hashes, then archive only this experiment. Any missing or changed file stops
packaging. The ZIP contains images and provenance for separate evaluation.

In [ ]:
import shutil

def validate_experiment(root):
    manifest = json.loads((root / "experiment.json").read_text(encoding="utf-8"))
    if manifest["status"] not in {"generated", "validated"}:
        raise RuntimeError(f"Experiment is incomplete: {manifest['status']}")
    labels = list(manifest["adapters"])
    if manifest["conditions_completed"] != labels:
        raise RuntimeError("Not all conditions completed.")
    expected = [f"gen_{i:03d}.png" for i in range(len(manifest["seeds"]))]
    settings = manifest["generation"]
    for label in labels:
        folder = root / label
        if {p.name for p in folder.glob("*.png")} != set(expected):
            raise RuntimeError(f"Missing or unexpected PNG files: {label}")
        records = json.loads((folder / "images.json").read_text(encoding="utf-8"))
        if [r["file"] for r in records] != expected or [r["seed"] for r in records] != manifest["seeds"]:
            raise RuntimeError(f"Image/seed records do not match the planned sequence: {label}")
        for index, record in enumerate(records):
            path = folder / record["file"]
            if record["index"] != index or record["adapter_sha256"] != manifest["adapters"][label]["sha256"]:
                raise RuntimeError(f"Incorrect pairing metadata: {path}")
            if sha256_file(path) != record["sha256"]:
                raise RuntimeError(f"Image hash mismatch: {path}")
            with Image.open(path) as image:
                if image.format != "PNG" or image.mode != "RGB" or image.size != (settings["width"], settings["height"]):
                    raise RuntimeError(f"Invalid output format or dimensions: {path}")
                image.verify()
    manifest["status"] = "validated"
    write_json(root / "experiment.json", manifest)
    return len(labels) * len(expected)

count = validate_experiment(EXPERIMENT_DIR)
archive = shutil.make_archive(str(EXPERIMENT_DIR), "zip", root_dir=EXPERIMENT_DIR)
print(f"Validated {count} images. Download: {archive}")

## Protocol notes

This notebook preserves the 512×512 study setting; it is not an assertion that
512 is SDXL's optimal resolution. Do not select different prompts, adapter weights,
steps or seeds after viewing individual conditions. Any comparison to an unadapted
base model should be a separately declared condition, not a substitute for a
clean-trained LoRA. Training-data provenance must be checked independently.

Fixed package versions and settings improve repeatability but do not establish
cross-device bitwise reproducibility or protection effectiveness. Changing CPU/GPU
RNG choice, dependencies, watermarking or numerical settings can change images
relative to older runs. Conditions may be generated separately; use the same
protocol and compare their saved manifests before treating the outputs as paired.

The code and control flow can be tested locally without SDXL weights. Full image
generation, runtime memory use and adapter compatibility still require a Kaggle GPU run.

References:
- [Diffusers reproducible pipelines](https://huggingface.co/docs/diffusers/v0.35.1/en/using-diffusers/reusing_seeds)
- [Single-file loading](https://huggingface.co/docs/diffusers/v0.35.1/en/api/loaders/single_file)
- [LoRA loading](https://huggingface.co/docs/diffusers/v0.35.1/en/api/loaders/lora)
- [DPM-Solver scheduler](https://huggingface.co/docs/diffusers/v0.35.1/en/api/schedulers/multistep_dpm_solver)